# Config

In [37]:
VERBOSE = True
DEBUG = False
PICKLE_VER = 2 # 1 - original version, 2 - version with GraphDataLibraryNoEmbedding converted to dict
GRAPH_SOURCE = 'directed' # directed, undirected graph data for building cluster (louvain, leiden)

In [38]:
if DEBUG:
    from models import *
    from gen_cluster_mitigationplan import *
    # import os, pickle
    # from .core import *

    dir_path = os.getcwd()
    # pcg_data_path = f"{dir_path}/graph_data_library_no_embedding.pkl"
    pcg_data_path = f"{dir_path}/graph_data_library_no_embedding_dict.pkl"
    # pcg_data_path = f"{dir_path}/graph_data_library.pkl"
    # graph_data_library.pkl
    # data = pickle.load(open(pcg_data_path, "rb"))

    # data: GraphDataLibraryNoEmbedding = pickle.load(open(pcg_data_path, "rb"))

    data_all = pickle.load(open(pcg_data_path, "rb"))
    # data_dict = data.model_dump()

    company_name = 'PCG|embedding_risk_desc_catalog|oneway_run'
    data = data_all['company_graph_datas'][company_name]
    debug_list = []
    all_data_list = []
# for company, data in company_graph_datas.items():
    G = nx.DiGraph()
    print(data.keys())
    nodes = data["nodes"]
    edges = data["edges"]
    print(edges[0].keys())
    line_weights = [edge["cosine_similarity"] for edge in edges]
    num_edges_to_show = get_number_edges_to_show(len(nodes))
    sorted_weights = sorted(line_weights, reverse=True)
    # The threshold is the weight of the (num_edges_to_show)-th edge (0-indexed)
    slider_value = sorted_weights[num_edges_to_show - 1]
    filter_edges = filter_non_arrow_edges2(edges, slider_value)
    for edge in filter_edges:
        G.add_edge(
            edge["source"],
            edge["target"],
            weight=edge["cosine_similarity"],
        )
        if edge["source"] == "risk_PCG_40":
            debug_list.append(edge)
    in_degree_centrality_dict = nx.in_degree_centrality(G)
    out_degree_centrality_dict = nx.out_degree_centrality(G)
    betweenness_dict_weight = nx.betweenness_centrality(G, weight="weight")
    betweenness_dict_non_weight = nx.betweenness_centrality(G)

    # create list of data so I can convert to dataframe later
    for node in nodes:
        row_data = {
            # "company": company,
            "risk_id": node["data"]["id"],
            "risk_name": node["data"]["label"],
            "risk_level": node["data"]["risk_level"],
            "in_degree": G.in_degree(node["data"]["id"]),
            "out_degree": G.out_degree(node["data"]["id"]),
            "in_degree_centrality": in_degree_centrality_dict.get(
                node["data"]["id"], None
            ),
            "out_degree_centrality": out_degree_centrality_dict.get(
                node["data"]["id"], None
            ),
            "betweenness_centrality_weight": betweenness_dict_weight.get(
                node["data"]["id"], None
            ),
            "betweenness_centrality_non_weight": betweenness_dict_non_weight.get(
                node["data"]["id"], None
            ),
        }
        all_data_list.append(row_data)

    all_data_df = pd.DataFrame(all_data_list)
    all_data_df

In [39]:
# edges

In [40]:
# filter_edges

In [41]:
# data_all['company_graph_datas']['PCG|embedding_risk_desc_catalog|oneway_run']['edges']

In [42]:
# data_all['company_graph_datas']['PCG|embedding_risk_desc_catalog|oneway_run']['nodes']

# load current version of pickle graph data

In [43]:
if PICKLE_VER == 2:
    from models import *
    from gen_cluster_mitigationplan import *
    # import os, pickle
    # from .core import *

    dir_path = os.getcwd()
    # pcg_data_path = f"{dir_path}/graph_data_library_no_embedding.pkl"
    pcg_data_path = f"{dir_path}/graph_data_library_no_embedding_dict.pkl"
    data = pickle.load(open(pcg_data_path, "rb"))

    data_source_selected = 'PCG|embedding_risk_desc_catalog|oneway_run' # 'lotus_south|embedding_risk_desc_catalog|oneway_run'
    all_data_df, clusters_directed, G, nodes = find_graph_properties_newpickle(pcg_data_path, data_source_selected)
    # all_data_df, clusters, G, nodes = find_graph_properties_newpickle(pcg_data_path, data_source_selected)

    clusters_undirected = find_clusters_from_graph(pcg_data_path, data_source_selected,CLUSTER_METHOD = 'leiden',graph_dir='undirected')
    clusters_directed = find_clusters_from_graph(pcg_data_path, data_source_selected,CLUSTER_METHOD = 'leiden',graph_dir='directed')

    if GRAPH_SOURCE == 'undirected':
        clusters = clusters_undirected
    else:
        clusters = clusters_directed


In [44]:
all_data_df

,risk_id,risk_name,risk_level,in_degree,out_degree,in_degree_centrality,out_degree_centrality,betweenness_centrality_weight,betweenness_centrality_non_weight
0,risk_PCG_20250513_0,Accounting errors,1,NaN,NaN,NaN,NaN,NaN,NaN
1,risk_PCG_20250513_1,Business interruption from fire hazards,2,0.0,1.0,0.000000,0.025641,0.0,0.000000
2,risk_PCG_20250513_2,Business interruption from flood,1,0.0,1.0,0.000000,0.025641,0.0,0.000000
3,risk_PCG_20250513_3,Business interruption from labor dispute,1,NaN,NaN,NaN,NaN,NaN,NaN
4,risk_PCG_20250513_4,Business interruption from natural disasters,2,1.0,0.0,0.025641,0.000000,0.0,0.000000
...,...,...,...,...,...,...,...,...,...
61,risk_PCG_20250513_61,Vehicles assets loss,2,NaN,NaN,NaN,NaN,NaN,NaN
62,risk_PCG_20250513_62,Vehicles failure and damage,2,0.0,2.0,0.000000,0.051282,0.0,0.000000
63,risk_PCG_20250513_63,Water pollution,2,1.0,0.0,0.025641,0.000000,0.0,0.000000
64,risk_PCG_20250513_64,Workforce shortage,2,1.0,0.0,0.025641,0.000000,0.0,0.000000


In [45]:
clusters

[['risk_PCG_20250513_54',
  'risk_PCG_20250513_65',
  'risk_PCG_20250513_41',
  'risk_PCG_20250513_59',
  'risk_PCG_20250513_32',
  'risk_PCG_20250513_24',
  'risk_PCG_20250513_37',
  'risk_PCG_20250513_64'],
 ['risk_PCG_20250513_40',
  'risk_PCG_20250513_48',
  'risk_PCG_20250513_46',
  'risk_PCG_20250513_36',
  'risk_PCG_20250513_39',
  'risk_PCG_20250513_33',
  'risk_PCG_20250513_28',
  'risk_PCG_20250513_21'],
 ['risk_PCG_20250513_45',
  'risk_PCG_20250513_58',
  'risk_PCG_20250513_60',
  'risk_PCG_20250513_10',
  'risk_PCG_20250513_43',
  'risk_PCG_20250513_42',
  'risk_PCG_20250513_56',
  'risk_PCG_20250513_62'],
 ['risk_PCG_20250513_12',
  'risk_PCG_20250513_29',
  'risk_PCG_20250513_25',
  'risk_PCG_20250513_55',
  'risk_PCG_20250513_18',
  'risk_PCG_20250513_1',
  'risk_PCG_20250513_51'],
 ['risk_PCG_20250513_11', 'risk_PCG_20250513_38', 'risk_PCG_20250513_35'],
 ['risk_PCG_20250513_2', 'risk_PCG_20250513_4'],
 ['risk_PCG_20250513_22', 'risk_PCG_20250513_63'],
 ['risk_PCG_2025

In [46]:
for i, j in enumerate(data['company_graph_datas'].keys()):
    print(i)
    print(j)

0
PCG|embedding_raw_user_data|oneway_run
1
PCG|embedding_risk_desc_catalog|oneway_run
2
PCG|embedding_summary_user_data|oneway_run
3
PCG|embedding_raw_user_data|twoway_run
4
PCG|embedding_risk_desc_catalog|twoway_run
5
PCG|embedding_summary_user_data|twoway_run
6
lotus_south|embedding_raw_user_data|oneway_run
7
lotus_south|embedding_risk_desc_catalog|oneway_run
8
lotus_south|embedding_summary_user_data|oneway_run
9
lotus_south|embedding_raw_user_data|twoway_run
10
lotus_south|embedding_risk_desc_catalog|twoway_run
11
lotus_south|embedding_summary_user_data|twoway_run


# load original version of pickle graph data

In [47]:
if PICKLE_VER == 1:
    from gen_cluster_mitigationplan import *
    from models import *

    # ==================== TEST CODE =====================
    dir_path = os.getcwd()
    # pcg_data_path = f"{dir_path}/graph_data_library_no_embedding.pkl"
    pcg_data_path = f"{dir_path}/nodes_and_edges_PCG.pkl"
    lotus_south_data_path = f"{dir_path}/nodes_and_edges_lotus_south.pkl"

    # data_path_dict = {"PCG": pcg_data_path, "Lotus South": lotus_south_data_path}
    data_path_dict = {"PCG": pcg_data_path}

    all_data_df, clusters, G, nodes = find_graph_properties(data_path_dict)


# find 'important' clusters

In [48]:
risk_high = all_data_df[all_data_df['risk_level']>=3] # TODO: need to handle the edge case when there's no high/critical risk level
# TopN risks with a given property (central/source/sink)
N_TOP = 3
# risk_central = top_n_with_row_limit(all_data_df, 'betweenness_centrality_non_weight', n=N_TOP)
# risk_source = top_n_with_row_limit(all_data_df, 'out_degree', n=N_TOP)
risk_central = top_n_strict_with_priority_on_highest(all_data_df, 'betweenness_centrality_non_weight', n=N_TOP)
risk_source = top_n_strict_with_priority_on_highest(all_data_df, 'out_degree', n=N_TOP)

# if VERBOSE:
#     print('high risk: ' + ', '.join(risk_high['risk_name'].tolist()))
#     print('central risk: ' + ', '.join(risk_central['risk_name'].tolist()))
#     print('source risk: ' + ', '.join(risk_source['risk_name'].tolist()))

# ======= cluster by type (high/central/source risk) =======

# cluster(s) that contains risk_high
highrisk_clusters = find_sublists_with_any(risk_high['risk_id'].tolist(), clusters)
# print(highrisk_clusters)

# cluster(s) that contains risk_central
centralrisk_clusters = find_sublists_with_any(risk_central['risk_id'].tolist(), clusters)
# print(centralrisk_clusters)

# cluster(s) that contains risk_source
sourcerisk_clusters = find_sublists_with_any(risk_source['risk_id'].tolist(), clusters)
# print(sourcerisk_clusters)

if VERBOSE:
    print('high risk: ' + ', '.join(risk_high['risk_name'].tolist()))
    for i, cluster in enumerate(highrisk_clusters, 1):
        print(f"\n--- Cluster {i} Prompt ---\n")
        print(generate_prompt_nointro(cluster, G, nodes))

    print('central risk: ' + ', '.join(risk_central['risk_name'].tolist()))
    for i, cluster in enumerate(centralrisk_clusters, 1):
        print(f"\n--- Cluster {i} Prompt ---\n")
        print(generate_prompt_nointro(cluster, G, nodes))

    print('source risk: ' + ', '.join(risk_source['risk_name'].tolist()))
    for i, cluster in enumerate(sourcerisk_clusters, 1):
        print(f"\n--- Cluster {i} Prompt ---\n")
        print(generate_prompt_nointro(cluster, G, nodes))

# ======= ALL important clusters (contains high/central/source risks) =======
combined_list = list(set(risk_high['risk_id'].tolist() + risk_central['risk_id'].tolist() + risk_source['risk_id'].tolist()))
important_clusters = find_sublists_with_any(combined_list, clusters)

# # === STEP 5: OUTPUT PROMPTS ===
print('============= Important Clusters =============')
for i, cluster in enumerate(important_clusters, 1):
    print(f"\n--- Cluster {i} Prompt ---\n")
    print(generate_prompt_nointro(cluster, G, nodes))

# print(generate_prompt_nointro(important_clusters[0], G, nodes))

high risk: Intense market competition

--- Cluster 1 Prompt ---

Risks:
- risk_20250513_54: Service-related dissatisfaction
- risk_20250513_65: Wrong delivery
- risk_20250513_41: Poor service quality
- risk_20250513_59: Uncompetitive service
- risk_20250513_32: New competitor into the market
- risk_20250513_24: Intense market competition
- risk_20250513_37: Outsourcing inefficiency
- risk_20250513_64: Workforce shortage

Dependencies:
- risk_20250513_65 -> risk_20250513_54 
- risk_20250513_24 -> risk_20250513_59 
- risk_20250513_41 -> risk_20250513_54 
- risk_20250513_41 -> risk_20250513_59 
- risk_20250513_41 -> risk_20250513_65 
- risk_20250513_32 -> risk_20250513_24 
- risk_20250513_37 -> risk_20250513_64 
- risk_20250513_37 -> risk_20250513_65 
- risk_20250513_37 -> risk_20250513_41 
- risk_20250513_54 -> risk_20250513_59 

central risk: Unable to deliver product, Poor service quality, Inventory damage

--- Cluster 1 Prompt ---

Risks:
- risk_20250513_54: Service-related dissatisfa

In [49]:
risk_high

,risk_id,risk_name,risk_level,in_degree,out_degree,in_degree_centrality,out_degree_centrality,betweenness_centrality_weight,betweenness_centrality_non_weight
24,risk_PCG_20250513_24,Intense market competition,4,1.0,1.0,0.025641,0.025641,0.000675,0.000675


# gen 'cluster' mitigation plan

In [ ]:
# from dotenv import load_dotenv
# import os
# from typing import List, Tuple

# TODO: need to acquire the api_key from your directory
# load_dotenv("../../.env")

estimate_cost.total_cost_THB = 0 # initialization
usage_count_list = []

# notice, this is a single cluster
cluster_risk_to_plan = generate_prompt_nointro(important_clusters[1], G, nodes)
# cluster_risk_to_plan = generate_prompt_nointro(highrisk_clusters, G, nodes)

# gen 'cluster' mitigation plans
response = get_response_control_cluster(cluster_risk_to_plan) # accept single cluster
# response = get_response_test(cluster_risk_to_plan)

usage_count = response.usage
usage_count_list.append(usage_count)    
MODEL = "gpt-4.1" # "gpt-4o"
estimate_cost(res_usage = usage_count_list,type='multi',model=MODEL)

In [ ]:
# response.choices[0].message.content

In [ ]:
# save to json
tmp_res = response.choices[0].message.content
report_json = json.loads(tmp_res)

data = []
data.append(report_json)

# input_file = report_json_folder + str(selected_year) + '_Q' + str(selected_quarter) + '_' + selected_company_report + '_json_riskcontrol.json'
file_path = f"{dir_path}/clustercontrol.json"

with open(file_path, 'w') as file:
        json.dump(data, file, indent=4, ensure_ascii=False)

report_json